In [0]:
%sql DROP SCHEMA IF EXISTS contraloria.employee_payroll CASCADE

# 📊 Payroll Time Series Analysis

## Objective
Analyze historical trends and patterns in the public employee payroll data from the Office of the Comptroller General of Panama.

## Data Source
**Table**: `contraloria.employee_payroll.bronze_contraloria_employees_scd_type2`

**SCD Type 2 Features:**
- `__START_AT`: Record validity start timestamp
- `__END_AT`: Record validity end timestamp (NULL = current record)
- `__ACTION`: Type of change (INSERT, UPDATE, DELETE)

## Analysis Focus
1. **Employee Count Trends**: Unique active employees over time
2. **Salary Evolution**: Total payroll budget changes
3. **Average Compensation**: Mean salary and allowance trends
4. **Distribution Analysis**: Salary ranges and percentiles
5. **Organizational Changes**: Hiring, terminations, and transfers

---

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Configuration
TABLE_NAME = "contraloria.employee_payroll.bronze_contraloria_employees_scd_type2"

print("✅ Setup complete")

In [0]:
# Load the SCD Type 2 table
df = spark.table(TABLE_NAME)

# Show schema
print("📋 Schema:")
df.printSchema()

# Basic statistics
total_records = df.count()
current_records = df.filter("__END_AT IS NULL").count()
historical_records = total_records - current_records

print(f"\n📊 Data Overview:")
print(f"  Total records (all versions): {total_records:,}")
print(f"  Current records: {current_records:,}")
print(f"  Historical records: {historical_records:,}")

# Date range
date_range = df.select(
    F.min("__START_AT").alias("earliest_date"),
    F.max(F.coalesce("__END_AT", F.current_timestamp())).alias("latest_date")
).collect()[0]

print(f"\n📅 Date Range:")
print(f"  Earliest: {date_range['earliest_date']}")
print(f"  Latest: {date_range['latest_date']}")

# Display sample records
print("\n🔍 Sample Records:")
display(df.orderBy(F.col("__START_AT").desc()).limit(10))

In [0]:
df.where('cedula =="1-0036-000516"').display()

In [0]:
df.groupBy(['cedula', 'institucion','fecha_actualizacion']).agg(F.countDistinct('cargo').alias('dupes')).display()

In [0]:
# Generate monthly time series of active employees
# For each month, count distinct employees who were active during that period

from pyspark.sql.functions import col, date_trunc, countDistinct, explode, sequence, lag, months_between
from pyspark.sql.window import Window

# Get date range
min_date = df.select(F.min("__START_AT")).collect()[0][0]
max_date = df.select(F.max(F.coalesce("__END_AT", F.current_timestamp()))).collect()[0][0]

# Create date spine (monthly buckets)
date_spine = spark.range(1).select(
    explode(
        sequence(
            date_trunc('month', F.lit(min_date)),
            date_trunc('month', F.lit(max_date)),
            F.expr("interval 1 month")
        )
    ).alias("month_start")
)

# Join with employee data to count active employees per month
active_by_month = (
    date_spine
    .join(
        df,
        (date_spine.month_start >= date_trunc('month', df.__START_AT)) &
        (date_spine.month_start < F.coalesce(df.__END_AT, F.current_timestamp())),
        "inner"
    )
    .groupBy("month_start")
    .agg(
        countDistinct("cedula").alias("unique_employees"),
        countDistinct("institucion").alias("unique_institutions")
    )
    .orderBy("month_start")
)

# Calculate monthly change using window function
window_spec = Window.orderBy("month_start")
monthly_employees = active_by_month.withColumn(
    "employee_change",
    col("unique_employees") - lag("unique_employees").over(window_spec)
)

print("📈 Monthly Active Employee Trends:")
display(monthly_employees)

# Convert to Pandas for visualization
employee_trend_pd = monthly_employees.toPandas()
employee_trend_pd['month_start'] = pd.to_datetime(employee_trend_pd['month_start'])

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Unique employees over time
ax1.plot(employee_trend_pd['month_start'], employee_trend_pd['unique_employees'], 
         marker='o', linewidth=2, color='#4d96ff')
ax1.set_title('Unique Active Employees Over Time', fontsize=14, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Number of Employees')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Monthly change
ax2.bar(employee_trend_pd['month_start'], employee_trend_pd['employee_change'].fillna(0), 
        color=['green' if x > 0 else 'red' for x in employee_trend_pd['employee_change'].fillna(0)])
ax2.set_title('Monthly Employee Change (Net Hiring/Termination)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Change in Employee Count')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n📊 Summary Statistics:")
print(f"  Average active employees: {employee_trend_pd['unique_employees'].mean():,.0f}")
print(f"  Peak employees: {employee_trend_pd['unique_employees'].max():,.0f}")
print(f"  Lowest employees: {employee_trend_pd['unique_employees'].min():,.0f}")

In [0]:
# Calculate total payroll costs over time (salary + allowance)

# Create date spine
min_date_val = df.select(F.min("__START_AT")).collect()[0][0]
max_date_val = df.select(F.max(F.coalesce("__END_AT", F.current_timestamp()))).collect()[0][0]

date_spine = spark.range(1).select(
    explode(
        sequence(
            date_trunc('month', F.lit(min_date_val)),
            date_trunc('month', F.lit(max_date_val)),
            F.expr("interval 1 month")
        )
    ).alias("month_start")
)

# Filter employees with valid salary
df_with_salary = df.filter(col("salario").isNotNull())

# Calculate monthly payroll
monthly_payroll = (
    date_spine
    .join(
        df_with_salary,
        (date_spine.month_start >= date_trunc('month', df_with_salary.__START_AT)) &
        (date_spine.month_start < F.coalesce(df_with_salary.__END_AT, F.current_timestamp())),
        "inner"
    )
    .groupBy("month_start")
    .agg(
        F.sum("salario").alias("total_salary"),
        F.sum("gasto").alias("total_allowance"),
        F.sum(col("salario") + F.coalesce(col("gasto"), F.lit(0))).alias("total_compensation"),
        countDistinct("cedula").alias("employee_count")
    )
    .withColumn("total_salary", F.round(col("total_salary"), 2))
    .withColumn("total_allowance", F.round(col("total_allowance"), 2))
    .withColumn("total_compensation", F.round(col("total_compensation"), 2))
    .withColumn("avg_compensation_per_employee", F.round(col("total_compensation") / col("employee_count"), 2))
    .orderBy("month_start")
)

payroll_evolution = monthly_payroll

print("💰 Monthly Payroll Budget Evolution:")
display(payroll_evolution)

# Visualize
payroll_pd = payroll_evolution.toPandas()
payroll_pd['month_start'] = pd.to_datetime(payroll_pd['month_start'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Total compensation over time
ax1.plot(payroll_pd['month_start'], payroll_pd['total_compensation'] / 1e6, 
         marker='o', linewidth=2, color='#6bcf7f', label='Total Compensation')
ax1.fill_between(payroll_pd['month_start'], 
                  payroll_pd['total_salary'] / 1e6, 
                  alpha=0.3, color='#4d96ff', label='Salary Only')
ax1.set_title('Total Monthly Payroll Budget (Millions USD)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Budget (Millions USD)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Average compensation per employee
ax2.plot(payroll_pd['month_start'], payroll_pd['avg_compensation_per_employee'], 
         marker='s', linewidth=2, color='#ff6b6b')
ax2.set_title('Average Compensation per Employee', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Average Compensation (USD)')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n💵 Budget Summary:")
print(f"  Current monthly payroll: ${payroll_pd['total_compensation'].iloc[-1]:,.2f}")
print(f"  Average monthly payroll: ${payroll_pd['total_compensation'].mean():,.2f}")
print(f"  Growth: {((payroll_pd['total_compensation'].iloc[-1] / payroll_pd['total_compensation'].iloc[0]) - 1) * 100:.1f}%")

In [0]:
# Analyze salary distribution for current employees

current_employees = df.filter("__END_AT IS NULL AND salario IS NOT NULL")

# Calculate percentiles
percentiles = current_employees.select(
    F.expr("percentile_approx(salario, array(0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99))").alias("percentiles")
).collect()[0][0]

print("📊 Salary Distribution (Current Employees):")
print(f"  10th percentile: ${percentiles[0]:,.2f}")
print(f"  25th percentile (Q1): ${percentiles[1]:,.2f}")
print(f"  50th percentile (Median): ${percentiles[2]:,.2f}")
print(f"  75th percentile (Q3): ${percentiles[3]:,.2f}")
print(f"  90th percentile: ${percentiles[4]:,.2f}")
print(f"  95th percentile: ${percentiles[5]:,.2f}")
print(f"  99th percentile: ${percentiles[6]:,.2f}")

# Salary ranges
salary_ranges = current_employees.select(
    F.when(F.col("salario") < 1000, "< $1,000")
    .when(F.col("salario") < 2000, "$1,000 - $2,000")
    .when(F.col("salario") < 3000, "$2,000 - $3,000")
    .when(F.col("salario") < 5000, "$3,000 - $5,000")
    .when(F.col("salario") < 10000, "$5,000 - $10,000")
    .otherwise("$10,000+").alias("salary_range")
).groupBy("salary_range").count().orderBy("salary_range")

print("\n💰 Salary Range Distribution:")
display(salary_ranges)

# Visualize
salary_data = current_employees.select("salario").toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax1.hist(salary_data['salario'], bins=50, color='#4d96ff', edgecolor='black', alpha=0.7)
ax1.axvline(percentiles[2], color='red', linestyle='--', linewidth=2, label=f'Median: ${percentiles[2]:,.0f}')
ax1.axvline(salary_data['salario'].mean(), color='green', linestyle='--', linewidth=2, 
            label=f'Mean: ${salary_data["salario"].mean():,.0f}')
ax1.set_title('Salary Distribution - Current Employees', fontsize=14, fontweight='bold')
ax1.set_xlabel('Salary (USD)')
ax1.set_ylabel('Number of Employees')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot
ax2.boxplot(salary_data['salario'], vert=True)
ax2.set_title('Salary Box Plot', fontsize=14, fontweight='bold')
ax2.set_ylabel('Salary (USD)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Analyze unique combinations and identify employees with multiple salaries

# 1. Clean data and get unique combinations
print("🧹 Data Cleaning & Unique Combinations Analysis")
print("="*80)

# Get all unique combinations of identifying fields
unique_combinations = (
    df
    .select("cedula", "nombre", "apellido", "cargo", "institucion", "salario", "__START_AT", "__END_AT")
    .filter(col("salario").isNotNull())
    .distinct()
)

total_unique = unique_combinations.count()
print(f"\n📊 Total unique combinations (cedula, nombre, apellido, cargo, institucion): {total_unique:,}")

# 2. Identify employees with multiple salary records over time (historical)
print("\n" + "="*80)
print("📈 EMPLOYEES WITH MULTIPLE SALARIES OVER TIME (Historical)")
print("="*80)

employee_salary_history = (
    df
    .filter(col("salario").isNotNull())
    .groupBy("cedula", "nombre", "apellido", "institucion")
    .agg(
        F.countDistinct("salario").alias("distinct_salaries"),
        F.count("*").alias("total_records"),
        F.min("salario").alias("min_salary"),
        F.max("salario").alias("max_salary"),
        F.avg("salario").alias("avg_salary"),
        F.collect_set("cargo").alias("positions_held")
    )
    .withColumn("salary_range", col("max_salary") - col("min_salary"))
    .withColumn("num_positions", F.size("positions_held"))
)

# Employees with multiple different salaries
multiple_salaries_historical = employee_salary_history.filter(col("distinct_salaries") > 1)

count_multiple_hist = multiple_salaries_historical.count()
total_employees = employee_salary_history.count()

print(f"\n📌 Summary:")
print(f"  • Total unique employees: {total_employees:,}")
print(f"  • Employees with multiple salary records: {count_multiple_hist:,}")
print(f"  • Percentage: {(count_multiple_hist / total_employees * 100):.1f}%")

print("\n🔍 Top 20 Employees with Most Salary Changes:")
display(
    multiple_salaries_historical
    .orderBy(col("distinct_salaries").desc(), col("salary_range").desc())
    .limit(20)
)

# 3. Current employees with multiple active salaries (potential data quality issues)
print("\n" + "="*80)
print("⚠️  CURRENT EMPLOYEES WITH MULTIPLE ACTIVE SALARIES")
print("="*80)

current_multiple_salaries = (
    df
    .filter((col("__END_AT").isNull()) & (col("salario").isNotNull()))
    .groupBy("cedula", "nombre", "apellido")
    .agg(
        F.countDistinct("salario").alias("distinct_salaries"),
        F.count("*").alias("active_records"),
        F.collect_list(F.struct("institucion", "cargo", "salario")).alias("salary_details")
    )
    .filter(col("distinct_salaries") > 1)
)

count_current_multiple = current_multiple_salaries.count()

print(f"\n📌 Current employees with multiple active salaries: {count_current_multiple:,}")

if count_current_multiple > 0:
    print("\n🚨 These employees may have data quality issues or work in multiple institutions:")
    display(current_multiple_salaries.orderBy(col("active_records").desc()).limit(50))
else:
    print("\n✅ No employees currently have multiple different salaries.")

# 4. Salary change tracking over time
print("\n" + "="*80)
print("💰 SALARY CHANGE TRACKING")
print("="*80)

employee_window = Window.partitionBy("cedula", "institucion").orderBy("__START_AT")

salary_changes = (
    df
    .filter(col("salario").isNotNull())
    .withColumn("previous_salary", lag("salario").over(employee_window))
    .withColumn("previous_cargo", lag("cargo").over(employee_window))
    .filter(col("previous_salary").isNotNull())
    .withColumn("salary_change", col("salario") - col("previous_salary"))
    .withColumn("salary_change_pct", F.round(((col("salario") / col("previous_salary")) - 1) * 100, 2))
    .withColumn("position_changed", col("cargo") != col("previous_cargo"))
    .filter(col("salary_change") != 0)
    .select(
        "cedula",
        F.concat(col("nombre"), F.lit(" "), col("apellido")).alias("full_name"),
        "institucion",
        "previous_cargo",
        "cargo",
        "previous_salary",
        col("salario").alias("new_salary"),
        "salary_change",
        "salary_change_pct",
        "position_changed",
        col("__START_AT").alias("change_date")
    )
)

total_changes = salary_changes.count()
print(f"\n📊 Total salary changes recorded: {total_changes:,}")

# Statistics
change_stats = salary_changes.select(
    F.avg("salary_change").alias("avg_change"),
    F.expr("percentile_approx(salary_change, 0.5)").alias("median_change"),
    F.max("salary_change").alias("max_increase"),
    F.min("salary_change").alias("max_decrease"),
    F.sum(F.when(col("salary_change") > 0, 1).otherwise(0)).alias("increases"),
    F.sum(F.when(col("salary_change") < 0, 1).otherwise(0)).alias("decreases")
).collect()[0]

print(f"\n📈 Change Statistics:")
print(f"  • Average change: ${change_stats['avg_change']:,.2f}")
print(f"  • Median change: ${change_stats['median_change']:,.2f}")
print(f"  • Largest increase: ${change_stats['max_increase']:,.2f}")
print(f"  • Largest decrease: ${change_stats['max_decrease']:,.2f}")
print(f"  • Salary increases: {change_stats['increases']:,}")
print(f"  • Salary decreases: {change_stats['decreases']:,}")

print("\n🔝 Top 20 Largest Salary Changes:")
display(salary_changes.orderBy(F.abs(col("salary_change")).desc()).limit(20))

# 5. Position changes
position_changes = salary_changes.filter(col("position_changed") == True)
print(f"\n🔄 Employees who changed positions: {position_changes.count():,}")
print("\nTop 10 position changes with salary impact:")
display(position_changes.orderBy(col("salary_change").desc()).limit(10))

In [0]:
# Analyze trends by institution over time

# Create date spine
min_date_val = df.select(F.min("__START_AT")).collect()[0][0]
max_date_val = df.select(F.max(F.coalesce("__END_AT", F.current_timestamp()))).collect()[0][0]

date_spine = spark.range(1).select(
    explode(
        sequence(
            date_trunc('month', F.lit(min_date_val)),
            date_trunc('month', F.lit(max_date_val)),
            F.expr("interval 1 month")
        )
    ).alias("month_start")
)

# Filter employees with valid salary
df_with_salary = df.filter(col("salario").isNotNull())

# Calculate metrics by institution and month
institution_monthly = (
    date_spine
    .join(
        df_with_salary,
        (date_spine.month_start >= date_trunc('month', df_with_salary.__START_AT)) &
        (date_spine.month_start < F.coalesce(df_with_salary.__END_AT, F.current_timestamp())),
        "inner"
    )
    .groupBy("month_start", "institucion")
    .agg(
        countDistinct("cedula").alias("employee_count"),
        F.sum(col("salario") + F.coalesce(col("gasto"), F.lit(0))).alias("total_compensation")
    )
    .withColumn("total_compensation", F.round(col("total_compensation"), 2))
    .withColumn("avg_compensation", F.round(col("total_compensation") / col("employee_count"), 2))
    .orderBy(col("month_start").desc(), col("employee_count").desc())
)

institution_trends = institution_monthly

print("🏛️ Employee Count by Institution Over Time:")
display(institution_trends)

# Top 10 institutions by current employee count
top_institutions = (
    df
    .filter((col("__END_AT").isNull()) & (col("salario").isNotNull()))
    .groupBy("institucion")
    .agg(
        F.count("*").alias("current_employees"),
        F.round(F.sum("salario"), 2).alias("total_salary"),
        F.round(F.sum("gasto"), 2).alias("total_allowance"),
        F.round(F.avg("salario"), 2).alias("avg_salary")
    )
    .orderBy(col("current_employees").desc())
    .limit(10)
)

print("\n📊 Top 10 Institutions by Employee Count (Current):")
display(top_institutions)

In [0]:
# Analyze if institutions are reporting consistently month-over-month

print("="*80)
print("🏛️ INSTITUTIONAL REPORTING CONSISTENCY ANALYSIS")
print("="*80)

# Get the range of report dates (fecha_consulta)
report_dates = (
    df
    .select("fecha_consulta")
    .distinct()
    .orderBy("fecha_consulta")
)

print("\n📅 Unique Report Dates (fecha_consulta):")
display(report_dates)

total_report_dates = report_dates.count()
print(f"\nTotal unique report dates: {total_report_dates}")

# Get all institutions that have ever reported
all_institutions = (
    df
    .select("institucion")
    .distinct()
    .orderBy("institucion")
)

total_institutions = all_institutions.count()
print(f"\nTotal unique institutions: {total_institutions}")

# Create a matrix of institution x report_date to check reporting consistency
institution_reporting = (
    df
    .groupBy("institucion", "fecha_consulta")
    .agg(
        F.countDistinct("cedula").alias("employee_count"),
        F.count("*").alias("record_count")
    )
    .orderBy("institucion", "fecha_consulta")
)

print("\n📊 Institution Reporting by Date:")
display(institution_reporting)

# Count how many reports each institution has submitted
institution_report_count = (
    institution_reporting
    .groupBy("institucion")
    .agg(
        F.countDistinct("fecha_consulta").alias("reports_submitted"),
        F.sum("employee_count").alias("total_employees_across_reports"),
        F.min("fecha_consulta").alias("first_report"),
        F.max("fecha_consulta").alias("last_report")
    )
    .withColumn("reporting_rate", F.round((col("reports_submitted") / F.lit(total_report_dates)) * 100, 1))
    .orderBy(col("reporting_rate").desc())
)

print("\n📈 Institution Reporting Summary:")
print(f"Expected reports per institution: {total_report_dates}")
display(institution_report_count)

# Identify institutions with missing reports
incomplete_reporters = institution_report_count.filter(col("reports_submitted") < total_report_dates)
incomplete_count = incomplete_reporters.count()

print(f"\n⚠️ Institutions with incomplete reporting: {incomplete_count} out of {total_institutions}")
if incomplete_count > 0:
    print("\nInstitutions missing some reports:")
    display(incomplete_reporters)

# Find gaps in reporting (which institutions missed which report dates)
print("\n🔍 Identifying Reporting Gaps...")

# Cross join all institutions with all report dates to get expected combinations
expected_reports = all_institutions.crossJoin(
    report_dates.select(col("fecha_consulta").alias("expected_date"))
)

# Left join with actual reports to find missing ones
reporting_gaps = (
    expected_reports
    .join(
        institution_reporting.select("institucion", col("fecha_consulta").alias("expected_date")),
        ["institucion", "expected_date"],
        "left_anti"  # Keep only rows that DON'T have a match (missing reports)
    )
    .orderBy("institucion", "expected_date")
)

total_gaps = reporting_gaps.count()
expected_total = total_institutions * total_report_dates

print(f"\n📊 Gap Analysis:")
print(f"  • Expected reports: {expected_total:,} (institutions × report dates)")
print(f"  • Actual reports: {institution_reporting.count():,}")
print(f"  • Missing reports (gaps): {total_gaps:,}")
print(f"  • Reporting completeness: {((expected_total - total_gaps) / expected_total * 100):.1f}%")

if total_gaps > 0:
    print(f"\n⚠️ Top 50 Missing Reports:")
    display(reporting_gaps.limit(50))

# Identify institutions that ONLY appear in old reports (potentially inactive)
latest_report_date = report_dates.agg(F.max("fecha_consulta")).collect()[0][0]

institutions_in_latest = (
    df
    .filter(col("fecha_consulta") == latest_report_date)
    .select("institucion")
    .distinct()
)

institutions_not_in_latest = (
    all_institutions
    .join(institutions_in_latest, "institucion", "left_anti")
)

missing_from_latest = institutions_not_in_latest.count()

print(f"\n🚨 Institutions NOT in the latest report ({latest_report_date}):")
print(f"  • Count: {missing_from_latest}")

if missing_from_latest > 0:
    print("\n⚠️ These institutions may have stopped reporting:")
    
    # Get their last report date
    last_seen = (
        df
        .join(institutions_not_in_latest, "institucion", "inner")
        .groupBy("institucion")
        .agg(
            F.max("fecha_consulta").alias("last_report_date"),
            F.countDistinct("cedula").alias("employees_in_last_report")
        )
        .orderBy(col("last_report_date").desc())
    )
    
    display(last_seen)

# Summary visualization data
print("\n" + "="*80)
print("📋 REPORTING CONSISTENCY SUMMARY")
print("="*80)

consistent_reporters = institution_report_count.filter(col("reports_submitted") == total_report_dates).count()
print(f"\n✅ Institutions reporting consistently: {consistent_reporters} ({(consistent_reporters/total_institutions*100):.1f}%)")
print(f"⚠️ Institutions with gaps: {incomplete_count} ({(incomplete_count/total_institutions*100):.1f}%)")
print(f"🚨 Institutions missing from latest report: {missing_from_latest}")

print("\n💡 RECOMMENDATIONS:")
print("  • Investigate institutions with reporting gaps")
print("  • Contact institutions missing from latest report")
print("  • Set up alerts for missing institutional reports")
print("  • Consider data completeness when calculating aggregates")
print("="*80)

In [0]:
# Generate summary insights from the analysis

print("="*80)
print("📋 KEY INSIGHTS - PAYROLL TIME SERIES ANALYSIS")
print("="*80)

# 1. Overall current state
current_stats = df.filter("__END_AT IS NULL").agg(
    F.count("*").alias("total_records"),
    F.countDistinct("cedula").alias("unique_employees"),
    F.sum("salario").alias("total_salary"),
    F.avg("salario").alias("avg_salary"),
    F.countDistinct("institucion").alias("total_institutions")
).collect()[0]

print(f"\n1️⃣ CURRENT STATE (Active Records):")
print(f"   • Total Active Records: {current_stats['total_records']:,}")
print(f"   • Unique Employees: {current_stats['unique_employees']:,}")
print(f"   • Total Institutions: {current_stats['total_institutions']}")
print(f"   • Total Monthly Payroll: ${current_stats['total_salary']:,.2f}")
print(f"   • Average Salary: ${current_stats['avg_salary']:,.2f}")

# 2. Historical data scope
historical_stats = df.agg(
    F.count("*").alias("total_historical_records"),
    F.countDistinct("cedula").alias("total_unique_employees"),
    F.min("__START_AT").alias("earliest_record"),
    F.max(F.coalesce("__END_AT", F.current_timestamp())).alias("latest_record")
).collect()[0]

print(f"\n2️⃣ HISTORICAL SCOPE:")
print(f"   • Total Historical Records: {historical_stats['total_historical_records']:,}")
print(f"   • All-time Unique Employees: {historical_stats['total_unique_employees']:,}")
print(f"   • Data Span: {historical_stats['earliest_record'].strftime('%Y-%m-%d')} to {historical_stats['latest_record'].strftime('%Y-%m-%d')}")

# 3. Data quality insights
null_salaries = df.filter("__END_AT IS NULL AND salario IS NULL").count()
duplicate_actives = df.filter("__END_AT IS NULL").groupBy("cedula", "institucion").count().filter("count > 1").count()

print(f"\n3️⃣ DATA QUALITY:")
print(f"   • Current records with missing salary: {null_salaries:,}")
print(f"   • Data completeness: {((current_stats['total_records'] - null_salaries) / current_stats['total_records'] * 100):.1f}%")
print(f"   • Employees with duplicate active records: {duplicate_actives:,}")

# 4. Salary change insights
employee_window = Window.partitionBy("cedula", "institucion").orderBy("__START_AT")
salary_change_count = (
    df
    .filter(col("salario").isNotNull())
    .withColumn("previous_salary", lag("salario").over(employee_window))
    .filter((col("previous_salary").isNotNull()) & (col("salario") != col("previous_salary")))
    .count()
)

print(f"\n4️⃣ SALARY CHANGES:")
print(f"   • Total recorded salary changes: {salary_change_count:,}")
print(f"   • Average changes per employee: {(salary_change_count / historical_stats['total_unique_employees']):.2f}")

# 5. Recommendations
print(f"\n5️⃣ RECOMMENDATIONS:")
print(f"   ✓ Monitor employees with multiple active salaries for data quality")
print(f"   ✓ Track salary change patterns for budget forecasting")
print(f"   ✓ Investigate large salary variations within same position/institution")
print(f"   ✓ Use SCD Type 2 history for compliance audits and compensation analysis")
print(f"   ✓ Clean duplicate active records to improve data integrity")

print("\n" + "="*80)
print("✅ Analysis Complete")
print("="*80)